In [1]:
import torch

x = torch.tensor([2.0, 4.0, 6.0])
y = torch.tensor([80.0, 140.0, 190.0])

assert x.dtype == torch.float32
assert y.dtype == torch.float32

w = 27.5
b = 26.67

y_hat = x * w + b
error = y - y_hat
mse = (error ** 2).mean()
assert abs(mse - 5.556) < 1e-3

print(mse)
print(mse.item())

tensor(5.5556)
5.555566787719727


In [2]:
import pandas as pd

train = pd.read_csv("../data/train_features.csv")
val = pd.read_csv("../data/val_features.csv")

X_train = train.drop(columns = ["price", "id"]).astype(float)
y_train = train["price"]
X_val = val.drop(columns = ["price", "id"]).astype(float)
y_val = val["price"]

X_train_std = (X_train - X_train.mean()) / X_train.std()
X_val_std = (X_val - X_train.mean()) / X_train.std()

X_tr_t = torch.tensor(X_train_std.to_numpy(), dtype = torch.float32)
y_tr_t = torch.tensor(y_train.to_numpy(), dtype = torch.float32)
X_va_t = torch.tensor(X_val_std.to_numpy(), dtype = torch.float32)
y_va_t = torch.tensor(y_val.to_numpy(), dtype = torch.float32)

assert X_tr_t.shape == (12747, 132)
assert y_tr_t.shape == (12747,)
assert X_va_t.shape == (2732, 132)
assert y_va_t.shape == (2732,)

assert X_tr_t.dtype == torch.float32
assert y_tr_t.dtype == torch.float32
assert X_va_t.dtype == torch.float32
assert y_va_t.dtype == torch.float32

assert max(abs(X_tr_t.mean(dim=0))) < 1e-4

In [3]:
from torch.utils.data import Dataset

class AirbnbDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        return (self.X[i], self.y[i])

train_ds = AirbnbDataset(X_tr_t, y_tr_t)
val_ds = AirbnbDataset(X_va_t, y_va_t)

assert len(train_ds) == 12747
assert len(val_ds) == 2732
assert train_ds[5][0].shape == (132,)
assert train_ds[3][1] == y_tr_t[3]
assert train_ds[len(train_ds) - 1][1] == y_tr_t[len(train_ds) - 1]
assert val_ds[8][1] == y_va_t[8]

In [4]:
# 64 * 199 + 11 = 12747 -> len(train_loader) = 200 and the last batch has 11 elements.

from torch.utils.data import DataLoader

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
assert len(train_loader) == 200
for xb, yb in train_loader:
    last = xb
assert len(last) == 11
for xb, yb in train_loader:
    assert xb.shape == (64, 132)
    assert yb.shape == (64,)
    break